In [ ]:
# Colab setup: clone the repo if needed, then install the extracted package.
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/haydenyoungcs/gradient-ascent.git"
REPO_DIR = pathlib.Path("/content/gradient-ascent")

github_token = os.environ.get("GITHUB_TOKEN")
wandb_api_key = os.environ.get("WANDB_API_KEY")

try:
    from google.colab import userdata  # type: ignore

    if github_token is None:
        github_token = userdata.get("GITHUB_TOKEN")
    if wandb_api_key is None:
        wandb_api_key = userdata.get("WANDB_API_KEY")
except Exception:
    pass

cwd = pathlib.Path.cwd()
project_root = cwd if (cwd / "pyproject.toml").exists() else REPO_DIR

if not project_root.exists():
    if github_token:
        clone_url = REPO_URL.replace("https://", f"https://{github_token}@")
        subprocess.run(["git", "clone", clone_url], check=True)
    else:
        raise RuntimeError(
            "Repo checkout not found. For this private repo, add a Colab secret or env var named "
            "GITHUB_TOKEN, or clone the repo manually before running this notebook."
        )

if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError(f"Expected pyproject.toml under {project_root}, but it was not found.")

os.chdir(project_root)
print(f"Changed working directory to {project_root}")

repo_src = project_root / "src"
for path in [project_root, repo_src]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(project_root), "wandb", "pot", "scikit-learn"], check=True)

import wandb

if wandb_api_key:
    wandb.login(key=wandb_api_key, relogin=True)
else:
    print("wandb installed; set WANDB_API_KEY if you want online logging.")


## 1-4. Setup, training, and core model checkpoints

This section trains the original and retrained models, then runs GA unlearning for the core checkpoint outputs. Redundant runtime/overall-bar figures are intentionally removed to keep outputs focused.

In [ ]:
import warnings

import torch
from IPython.display import Image as IPyImage, display

from gradient_ascent.data import default_num_workers, load_cifar10_datasets
from gradient_ascent.experiments import CoreExperimentConfig, ensure_wandb_run, run_core_checkpoints
from gradient_ascent.models import Net
from gradient_ascent.training import build_amp_config, configure_runtime

try:
    import wandb
except ImportError:
    wandb = None

warnings.filterwarnings("ignore", category=DeprecationWarning)

NUM_CLASSES = 10
OUT_DIR = "out"
resnet_model_depth = 50

configure_runtime()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_config = build_amp_config(device)
USE_BF16 = amp_config.dtype == torch.bfloat16

trainset, testset = load_cifar10_datasets(root="./data")
use_cuda = device.type == "cuda"
num_workers = default_num_workers(use_cuda)
model_factory = lambda: Net(num_classes=NUM_CLASSES, pretrained=False, model_depth=resnet_model_depth).to(device)

core_config = CoreExperimentConfig(
    num_classes=NUM_CLASSES,
    model_depth=resnet_model_depth,
    out_dir=OUT_DIR,
    unlearn_batch_size=512 if USE_BF16 else 256,
)

wandb_run = ensure_wandb_run(
    wandb,
    project="gradient-ascent",
    name="core-checkpoints",
    config={
        "target_label": core_config.target_label,
        "num_epochs": core_config.num_epochs,
        "batch_size": core_config.batch_size,
        "model_depth": core_config.model_depth,
    },
)

core_artifacts = run_core_checkpoints(
    model_factory=model_factory,
    trainset=trainset,
    testset=testset,
    device=device,
    use_cuda=use_cuda,
    num_workers=num_workers,
    config=core_config,
    wandb_run=wandb_run,
    wandb_module=wandb,
)

display(IPyImage(filename=core_artifacts.original_vs_retrain_plot_path))
for algorithm_key in ["ga", "ssd", "salun"]:
    artifact = core_artifacts.algorithm_artifacts[algorithm_key]
    display(IPyImage(filename=artifact.classwise_percent_plot_path))
    display(IPyImage(filename=artifact.classwise_absolute_plot_path))

for key, value in core_artifacts.summary_metrics.items():
    print(f"{key}: {value:.3f}")


## 5. Shared similarity helpers

Defines reusable activation extraction, metric evaluation, and plotting helpers used by downstream trajectory analysis. Standalone final pairwise snapshot generation is removed to avoid redundancy.

In [ ]:
# Shared similarity helpers are imported from the reusable package.
from gradient_ascent.models import DEFAULT_LAYER_NAMES
from gradient_ascent.similarity import (
    CCA,
    CKA,
    CosineSimilarity,
    EarthMoversDistance,
    EuclideanDistance,
    GromovWassersteinDistance,
    HIGHER_BETTER_METRICS,
    KLDivergence,
    LOWER_BETTER_METRICS,
    build_default_metrics,
    collect_model_activations,
    evaluate_pair_rows,
    plot_grouped_bars,
    transform_rows_for_plot,
)

layer_names = list(DEFAULT_LAYER_NAMES)
metrics = build_default_metrics()
higher_better_metrics = list(HIGHER_BETTER_METRICS)
lower_better_metrics = list(LOWER_BETTER_METRICS)
plot_metric_names = higher_better_metrics + lower_better_metrics

print("Shared similarity helpers and metrics loaded from gradient_ascent.similarity.")

## 6. Epoch-wise unlearning trajectories, MIA, and animations

Runs GA, SSD, and SalUn snapshot trajectories, computes MIA trajectories, and generates evolving similarity summaries and GIFs for both references: unlearned-vs-retrained and unlearned-vs-original.

In [ ]:
# Unlearning algorithm comparison trajectories (GA vs SSD vs SalUn).
# Computes artifact files in the package, then displays them compactly here.

from IPython.display import Image as IPyImage, display

from gradient_ascent.experiments import TrajectoryExperimentConfig, ensure_wandb_run, run_trajectory_analysis

if "collect_model_activations" not in globals() or "evaluate_pair_rows" not in globals() or "transform_rows_for_plot" not in globals():
    raise RuntimeError("Run the shared similarity helper cell before this cell.")

trajectory_config = TrajectoryExperimentConfig(
    num_classes=NUM_CLASSES,
    model_depth=resnet_model_depth,
    out_dir=OUT_DIR,
    trajectory_batch_size=512 if USE_BF16 else 256,
)

trajectory_artifacts = run_trajectory_analysis(
    model_factory=model_factory,
    trainset=trainset,
    testset=testset,
    device=device,
    use_cuda=use_cuda,
    num_workers=num_workers,
    config=trajectory_config,
    original_checkpoint_path=f"{OUT_DIR}/original_net.pt",
    retrained_checkpoint_path=f"{OUT_DIR}/retrained_from_scratch_net.pt",
    snapshot_dirs={
        "ga": f"{OUT_DIR}/unlearning_snapshots_ga",
        "ssd": f"{OUT_DIR}/unlearning_snapshots_ssd",
        "salun": f"{OUT_DIR}/unlearning_snapshots_salun",
    },
    layer_names=layer_names,
    metric_names=plot_metric_names,
    lower_better_metrics=lower_better_metrics,
    activation_collector=lambda model, loader: collect_model_activations(
        model,
        loader,
        layer_names,
        device,
        max_batches=trajectory_config.max_batches_for_similarity,
    ),
    pair_evaluator=evaluate_pair_rows,
    transform_rows_for_plot=transform_rows_for_plot,
    wandb_run=ensure_wandb_run(wandb, project="gradient-ascent", name="unlearning-algorithm-comparison"),
    wandb_module=wandb,
)

for algorithm_key in ["ga", "ssd", "salun"]:
    mia_artifact = trajectory_artifacts.mia_artifacts[algorithm_key]
    display(IPyImage(filename=mia_artifact.grid_plot_path))
    display(IPyImage(filename=mia_artifact.control_plot_path))
    for reference_key in ["retrained", "original"]:
        similarity_artifact = trajectory_artifacts.similarity_artifacts[algorithm_key][reference_key]
        display(IPyImage(filename=similarity_artifact.summary_plot_path))
        display(IPyImage(filename=similarity_artifact.gif_path))


## 7. Combined cross-algorithm trajectory comparison

Overlays GA, SSD, and SalUn similarity + MIA trajectories in a single consolidated figure for direct comparison.

In [ ]:
# Integrated combined comparison figure: similarity + MIA trajectories.
# Reads saved trajectory CSV artifacts and displays the final summary inline.

from IPython.display import Image as IPyImage, display

from gradient_ascent.experiments import CombinedComparisonConfig, ensure_wandb_run, save_combined_trajectory_comparison

combined_path = save_combined_trajectory_comparison(
    CombinedComparisonConfig(out_dir=OUT_DIR),
    wandb_run=ensure_wandb_run(wandb, project="gradient-ascent", name="unlearning-algorithm-comparison"),
    wandb_module=wandb,
)
print(f"Saved integrated comparison figure to {combined_path}")
display(IPyImage(filename=combined_path))
